In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.preprocessing import load_data, split_data, get_preprocessor, RANDOM_STATE
from src.models import get_models

print("Đang chạy Cross-Validation...")

# 1. Load và chia dữ liệu
df = load_data()
X_train, X_test, y_train, y_test = split_data(df)

# 2. Thiết lập 5-Fold Stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["recall", "precision", "f1", "roc_auc"]

rows = []
# 3. Đánh giá từng mô hình
for name, model in get_models().items():
    pipe = ImbPipeline([
        ("preprocessor", get_preprocessor()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model)
    ])
    
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    
    row = {"Model": name}
    for m in scoring:
        row[f"{m}_mean"] = res[f"test_{m}"].mean()
        row[f"{m}_std"] = res[f"test_{m}"].std()
    rows.append(row)
    print(f" -> Hoàn tất mô hình: {name}")

# 4. Xuất bảng kết quả ra file
cv_results = pd.DataFrame(rows).sort_values(by="recall_mean", ascending=False).round(3)
cv_results.to_csv("results/cv_results.csv", index=False)

print("\n=== XUẤT KẾT QUẢ THÀNH CÔNG ===")
display(cv_results)